In [1]:
!pip install mysql-connector-python ipython-sql sqlalchemy pandas

Defaulting to user installation because normal site-packages is not writeable


In [2]:
%load_ext sql 
from sqlalchemy import create_engine
import pandas as pd 

In [3]:
user = "root"  # e.g., root
password = "root"  # MySQL Workbench password
host = "127.0.0.1"  # or localhost
port = 3306  # Default MySQL port
database = "RideSharingSystem"  # Database to connect # Create connection string
connection_string = f"mysql+mysqlconnector://{user}:{password}@{host}:{port}"
        
from sqlalchemy import create_engine, text
engine = create_engine(connection_string)

# Create the database using SQLAlchemy's text() function
with engine.connect() as conn:
    conn.execute(text("CREATE DATABASE IF NOT EXISTS RideSharingSystem"))
    print("Database created successfully!")

Database created successfully!


In [4]:
from sqlalchemy import create_engine, inspect, text

# Create connection string
connection_string = f"mysql+mysqlconnector://{user}:{password}@{host}:{port}/{database}"

# Create engine
engine = create_engine(connection_string)

# Function to drop foreign key constraints before dropping the table
def drop_foreign_keys_and_table(engine, table_name):
    try:
        # Use the connection to execute the operations
        with engine.connect() as connection:
            # Check if the table exists using INFORMATION_SCHEMA
            table_exists_query = f"""
                SELECT TABLE_NAME 
                FROM INFORMATION_SCHEMA.TABLES 
                WHERE TABLE_SCHEMA = '{database}' AND TABLE_NAME = '{table_name}'
            """
            table_exists = connection.execute(text(table_exists_query)).fetchone()

            if not table_exists:
                print(f"Table '{table_name}' does not exist.")
                return
            
            # Identify foreign keys referencing this table
            referencing_fks_query = f"""
                SELECT TABLE_NAME, CONSTRAINT_NAME 
                FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE 
                WHERE REFERENCED_TABLE_NAME = '{table_name}' 
                  AND REFERENCED_TABLE_SCHEMA = '{database}'
            """
            referencing_fks = connection.execute(text(referencing_fks_query)).fetchall()

            # Drop foreign keys in referencing tables
            for fk in referencing_fks:
                ref_table = fk[0]
                fk_name = fk[1]
                try:
                    drop_fk_query = f"ALTER TABLE {ref_table} DROP FOREIGN KEY {fk_name}"
                    connection.execute(text(drop_fk_query))
                    print(f"Dropped foreign key {fk_name} from table '{ref_table}'.")
                except Exception as err:
                    print(f"Error dropping foreign key {fk_name} from table '{ref_table}': {err}")
            
            # Drop foreign key constraints in this table
            fk_query = f"""
                SELECT CONSTRAINT_NAME 
                FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE 
                WHERE TABLE_NAME = '{table_name}' 
                  AND TABLE_SCHEMA = '{database}'
                  AND REFERENCED_TABLE_NAME IS NOT NULL
            """
            foreign_keys = connection.execute(text(fk_query)).fetchall()

            for fk in foreign_keys:
                fk_name = fk[0]
                try:
                    drop_fk_query = f"ALTER TABLE {table_name} DROP FOREIGN KEY {fk_name}"
                    connection.execute(text(drop_fk_query))
                    print(f"Dropped foreign key {fk_name} from table '{table_name}'.")
                except Exception as err:
                    print(f"Error dropping foreign key {fk_name} from table '{table_name}': {err}")

            # Drop the primary key if needed
            pk_query = f"SHOW KEYS FROM {table_name} WHERE Key_name = 'PRIMARY'"
            primary_key = connection.execute(text(pk_query)).fetchone()
            if primary_key:
                drop_pk_query = f"ALTER TABLE {table_name} DROP PRIMARY KEY"
                connection.execute(text(drop_pk_query))
                print(f"Dropped primary key from table '{table_name}'.")

            # Now drop the table
            drop_table_query = f"DROP TABLE {table_name}"
            connection.execute(text(drop_table_query))
            print(f"Table '{table_name}' dropped successfully.")

    except Exception as err:
        print(f"Error while dropping table '{table_name}': {err}")

# List of tables in reverse order of dependency (child tables first, parent tables last)
tables = [
    "Rating",
    "TripBooking",
    "RideHistory",
    "Payment",
    "Driver",
    "Passenger",
    "VehicleDetails"
]

# Loop through the list and drop each table after removing foreign key constraints
for table in tables:
    drop_foreign_keys_and_table(engine, table)

# Engine will automatically handle connection closing


Dropped foreign key rating_ibfk_1 from table 'Rating'.
Dropped foreign key rating_ibfk_2 from table 'Rating'.
Dropped foreign key rating_ibfk_3 from table 'Rating'.
Dropped primary key from table 'Rating'.
Table 'Rating' dropped successfully.
Dropped foreign key ridehistory_ibfk_1 from table 'ridehistory'.
Dropped foreign key payment_ibfk_1 from table 'payment'.
Dropped foreign key tripbooking_ibfk_1 from table 'TripBooking'.
Dropped foreign key tripbooking_ibfk_2 from table 'TripBooking'.
Dropped foreign key tripbooking_ibfk_3 from table 'TripBooking'.
Dropped primary key from table 'TripBooking'.
Table 'TripBooking' dropped successfully.
Dropped primary key from table 'RideHistory'.
Table 'RideHistory' dropped successfully.
Dropped foreign key payment_ibfk_2 from table 'Payment'.
Dropped primary key from table 'Payment'.
Table 'Payment' dropped successfully.
Dropped foreign key driver_ibfk_1 from table 'Driver'.
Dropped primary key from table 'Driver'.
Table 'Driver' dropped successf

In [5]:
from sqlalchemy import text

create_vehicle_details_query = """
CREATE TABLE IF NOT EXISTS VehicleDetails (
    VehicleID INT PRIMARY KEY,
    LicensePlateNum VARCHAR(15) NOT NULL,
    Model VARCHAR(20) NOT NULL,
    Capacity INT NOT NULL,
    VehicleColor VARCHAR(15) NOT NULL
);
"""

try:
    with engine.connect() as conn:
        conn.execute(text(create_vehicle_details_query))
        print("VehicleDetails table created successfully!")
except Exception as e:
    print("Error creating VehicleDetails table:", e)


VehicleDetails table created successfully!


In [6]:
create_driver_query = """
CREATE TABLE IF NOT EXISTS Driver (
    DriverID INT PRIMARY KEY,
    Name VARCHAR(20) NOT NULL,
    Contact VARCHAR(15) NOT NULL,
    VehicleID INT NOT NULL,
    AvgDriverRating DECIMAL(2,1) NOT NULL,
    AvailabilityStatus ENUM('Available', 'Unavailable') NOT NULL,
    DriverAccountStatus ENUM('Active', 'Suspended', 'Banned') NOT NULL,
    FOREIGN KEY (VehicleID) REFERENCES VehicleDetails(VehicleID)
);
"""

try:
    with engine.connect() as conn:
        conn.execute(text(create_driver_query))
        print("Driver table created successfully!")
except Exception as e:
    print("Error creating Driver table:", e)


Driver table created successfully!


In [7]:
create_passenger_query = """
CREATE TABLE IF NOT EXISTS Passenger (
    PassengerID INT PRIMARY KEY,
    Name VARCHAR(20) NOT NULL,
    Contact VARCHAR(15) NOT NULL,
    PassengerAccountStatus ENUM('Active', 'Suspended', 'Banned') NOT NULL
);
"""

try:
    with engine.connect() as conn:
        conn.execute(text(create_passenger_query))
        print("Passenger table created successfully!")
except Exception as e:
    print("Error creating Passenger table:", e)


Passenger table created successfully!


In [8]:
create_trip_booking_query = """
CREATE TABLE IF NOT EXISTS TripBooking (
    TripID INT PRIMARY KEY,
    DriverID INT NOT NULL,
    PassengerID INT NOT NULL,
    VehicleID INT NOT NULL,
    PickupLocation VARCHAR(255) NOT NULL,
    DropLocation VARCHAR(255) NOT NULL,
    TripStatus ENUM('Pending', 'InProgress', 'Completed') NOT NULL,
    TripBookingTime DATETIME NOT NULL,
    RideStartTime DATETIME NOT NULL,
    RideEndTime DATETIME NOT NULL,
    Fare DECIMAL(10,2) NOT NULL,
    Tip DECIMAL(10,2) NOT NULL,
    FOREIGN KEY (DriverID) REFERENCES Driver(DriverID),
    FOREIGN KEY (PassengerID) REFERENCES Passenger(PassengerID),
    FOREIGN KEY (VehicleID) REFERENCES VehicleDetails(VehicleID)
);
"""

try:
    with engine.connect() as conn:
        conn.execute(text(create_trip_booking_query))
        print("TripBooking table created successfully!")
except Exception as e:
    print("Error creating TripBooking table:", e)


TripBooking table created successfully!


In [10]:
create_ride_history_query = """
CREATE TABLE IF NOT EXISTS RideHistory (
    HistoryID INT PRIMARY KEY,
    TripID INT NOT NULL,
    RideStatus ENUM('Inprogress', 'Completed', 'Pending') NOT NULL,
    FOREIGN KEY (TripID) REFERENCES TripBooking(TripID)
);
"""

try:
    with engine.connect() as conn:
        conn.execute(text(create_ride_history_query))
        print("RideHistory table created successfully!")
except Exception as e:
    print("Error creating RideHistory table:", e)


RideHistory table created successfully!


In [11]:
create_payment_query = """
CREATE TABLE IF NOT EXISTS Payment (
    PaymentID INT PRIMARY KEY,
    TripID INT NOT NULL,
    PassengerID INT NOT NULL,
    PaymentMethod ENUM('Credit', 'Debit') NOT NULL,
    PaymentStatus ENUM('Completed', 'Pending', 'Failed') NOT NULL,
    Amount DECIMAL(10,2) NOT NULL,
    FOREIGN KEY (TripID) REFERENCES TripBooking(TripID),
    FOREIGN KEY (PassengerID) REFERENCES Passenger(PassengerID)
);
"""

try:
    with engine.connect() as conn:
        conn.execute(text(create_payment_query))
        print("Payment table created successfully!")
except Exception as e:
    print("Error creating Payment table:", e)


Payment table created successfully!


In [12]:
create_rating_query = """
CREATE TABLE IF NOT EXISTS Rating (
    RatingID INT PRIMARY KEY,
    DriverID INT NOT NULL,
    PassengerID INT NOT NULL,
    TripID INT NOT NULL,
    RatingScore DECIMAL(2,1) NOT NULL,
    Feedback VARCHAR(255),
    FOREIGN KEY (DriverID) REFERENCES Driver(DriverID),
    FOREIGN KEY (PassengerID) REFERENCES Passenger(PassengerID),
    FOREIGN KEY (TripID) REFERENCES TripBooking(TripID)
);
"""

try:
    with engine.connect() as conn:
        conn.execute(text(create_rating_query))
        print("Rating table created successfully!")
except Exception as e:
    print("Error creating Rating table:", e)


Rating table created successfully!


In [13]:
show_tables_query = "SHOW TABLES;"

try:
    with engine.connect() as conn:
        result = conn.execute(text(show_tables_query))
        print("Tables in the database:")
        for row in result:
            print(row[0])
except Exception as e:
    print("Error listing tables:", e)


Tables in the database:
driver
passenger
payment
rating
ridehistory
tripbooking
vehicledetails


In [14]:
# Reusable function to fetch records using any SQL query
def fetch_data(query):
    try:
        # Execute the query and load the results into a pandas DataFrame
        data = pd.read_sql(query, engine)
        return data
    except Exception as e:
        print(f"Error occurred: {e}")
        return None  # Return None if there is an error

In [15]:

# Inserting data into Passenger table
insert_passenger_query = """
INSERT INTO Passenger (PassengerID, Name, Contact, PassengerAccountStatus)
VALUES
(11, 'David Lee', '555-1111', 'Active'),
(12, 'Rachel Adams', '555-2222', 'Banned'),
(13, 'Tom Harris', '555-3333', 'Active'),
(14, 'Sophia Green', '555-4444', 'Active'),
(15, 'James White', '555-5555', 'Banned'),
(16, 'Olivia Black', '555-6666', 'Active'),
(17, 'Emma Brown', '555-7777', 'Active'),
(18, 'Liam Smith', '555-8888', 'Banned'),
(19, 'Noah Johnson', '555-9999', 'Active'),
(20, 'Ava Jones', '555-1010', 'Active'),
(21, 'William Garcia', '555-1212', 'Banned'),
(22, 'Mia Martinez', '555-1313', 'Active'),
(23, 'Lucas Rodriguez', '555-1414', 'Active'),
(24, 'Ella Davis', '555-1515', 'Banned'),
(25, 'Benjamin Wilson', '555-1616', 'Active'),
(26, 'Charlotte Moore', '555-1717', 'Active'),
(27, 'Henry Taylor', '555-1818', 'Banned'),
(28, 'Amelia Anderson', '555-1919', 'Active'),
(29, 'Elijah Thomas', '555-2020', 'Active'),
(30, 'Isabella Hernandez', '555-2121', 'Banned'),
(31, 'Logan Martin', '555-2222', 'Active'),
(32, 'Sofia Lee', '555-2323', 'Active'),
(33, 'Jackson Perez', '555-2424', 'Banned'),
(34, 'Harper Clark', '555-2525', 'Active'),
(35, 'Sebastian Lewis', '555-2626', 'Active'),
(36, 'Evelyn Young', '555-2727', 'Banned'),
(37, 'Alexander King', '555-2828', 'Active'),
(38, 'Lily Hall', '555-2929', 'Active'),
(39, 'Daniel Wright', '555-3030', 'Banned'),
(40, 'Grace Lopez', '555-3131', 'Active'),
(41, 'Matthew Hill', '555-3232', 'Active'),
(42, 'Scarlett Scott', '555-3333', 'Banned'),
(43, 'Joseph Green', '555-3434', 'Active'),
(44, 'Chloe Adams', '555-3535', 'Active'),
(45, 'Samuel Nelson', '555-3636', 'Banned'),
(46, 'Victoria Baker', '555-3737', 'Active');
"""

# Try inserting data
try:
    with engine.connect() as conn:
        conn.execute(text(insert_passenger_query))
        conn.commit()  # Ensure changes are committed to the database
        print("records inserted successfully into Passenger.")
except Exception as e:
    print("Error inserting data into Passenger:", e)


records inserted successfully into Passenger.


In [16]:
    
# Fetch all records from the Passenger table
fetch_query = "SELECT * FROM Passenger"
passenger_data = fetch_data(fetch_query)

# Display the fetched records
if passenger_data is not None:
    display(passenger_data)
else:
    print("Failed to fetch data.")

,PassengerID,Name,Contact,PassengerAccountStatus
0,11,David Lee,555-1111,Active
1,12,Rachel Adams,555-2222,Banned
2,13,Tom Harris,555-3333,Active
3,14,Sophia Green,555-4444,Active
4,15,James White,555-5555,Banned
5,16,Olivia Black,555-6666,Active
6,17,Emma Brown,555-7777,Active
7,18,Liam Smith,555-8888,Banned
8,19,Noah Johnson,555-9999,Active
9,20,Ava Jones,555-1010,Active


In [17]:
# Inserting records into VehicleDetails table
insert_query = """
INSERT INTO VehicleDetails (VehicleID, LicensePlateNum, Model, Capacity, VehicleColor)
VALUES 
(1, 'TX53JJ3667', 'Corolla', 6, 'Red'),
(2, 'TX45ZZ1234', 'Honda Civic', 5, 'Blue'),
(3, 'AB12CD5678', 'Ford Mustang', 4, 'Black'),
(4, 'XY98KL2345', 'Chevrolet Malibu', 6, 'White'),
(5, 'AA12BB3456', 'BMW 3 Series', 4, 'Silver'),
(6, 'GG32FF7890', 'Tesla Model 3', 5, 'Red'),
(7, 'ZZ11RR6789', 'Toyota Camry', 5, 'Green'),
(8, 'LL22UU9876', 'Honda Accord', 6, 'Black'),
(9, 'MM33VV8765', 'Nissan Altima', 4, 'Blue'),
(10, 'PP44WW7654', 'Ford Focus', 5, 'Red'),
(11, 'QQ55XX6543', 'Chevrolet Impala', 6, 'Silver'),
(12, 'RR66YY5432', 'Hyundai Elantra', 4, 'White'),
(13, 'SS77ZZ4321', 'Mazda 3', 5, 'Yellow'),
(14, 'TT88AA3210', 'Kia Optima', 6, 'Purple'),
(15, 'UU99BB2109', 'Subaru Impreza', 4, 'Orange'),
(16, 'VV00CC1098', 'Volkswagen Jetta', 5, 'Pink'),
(17, 'WW11DD0987', 'BMW 5 Series', 6, 'Brown'),
(18, 'XX22EE9876', 'Audi A4', 4, 'Gold'),
(19, 'YY33FF8765', 'Mercedes-Benz', 5, 'Grey'),
(20, 'ZZ44GG7654', 'Lexus IS', 6, 'Red'),
(21, 'AA55HH6543', 'Chevrolet Camaro', 5, 'Blue'),
(22, 'BB66II5432', 'Porsche 911', 4, 'Black'),
(23, 'CC77JJ4321', 'Jaguar XF', 6, 'Silver'),
(24, 'DD88KK3210', 'Lincoln MKZ', 5, 'White'),
(25, 'EE99LL2109', 'Cadillac CTS', 4, 'Green'),
(26, 'FF00MM1098', 'Mitsubishi Outlander', 5, 'Purple'),
(27, 'GG11NN0987', 'Chrysler 300', 4, 'Red'),
(28, 'HH22OO9876', 'Buick Regal', 5, 'Yellow'),
(29, 'II33PP8765', 'Volvo S60', 6, 'Orange'),
(30, 'JJ44QQ7654', 'Infiniti Q50', 4, 'Pink');

"""

# Using try-except block to handle errors
try:
    # Connect to the database and execute the query
    with engine.connect() as conn:
        conn.execute(text(insert_query))
        conn.commit()  # Commit the transaction
    print("Records inserted successfully into the VehicleDetails table.")
except Exception as e:
    # Capture any general exceptions
    print(f"An unexpected error occurred: {e}")

Records inserted successfully into the VehicleDetails table.


In [18]:
# Fetch all records from the VehicleDetails table
fetch_query = "SELECT * FROM VehicleDetails"
vehicledetails_data = fetch_data(fetch_query)

# Display the fetched records
if vehicledetails_data is not None:
    display(vehicledetails_data)
else:
    print("Failed to fetch data.")

,VehicleID,LicensePlateNum,Model,Capacity,VehicleColor
0,1,TX53JJ3667,Corolla,6,Red
1,2,TX45ZZ1234,Honda Civic,5,Blue
2,3,AB12CD5678,Ford Mustang,4,Black
3,4,XY98KL2345,Chevrolet Malibu,6,White
4,5,AA12BB3456,BMW 3 Series,4,Silver
5,6,GG32FF7890,Tesla Model 3,5,Red
6,7,ZZ11RR6789,Toyota Camry,5,Green
7,8,LL22UU9876,Honda Accord,6,Black
8,9,MM33VV8765,Nissan Altima,4,Blue
9,10,PP44WW7654,Ford Focus,5,Red


In [19]:
# SQL query to insert driver records
insert_query_driver = """
INSERT INTO Driver (DriverID, Name, Contact, VehicleID, AvgDriverRating, AvailabilityStatus, DriverAccountStatus)
VALUES 
(111, 'John Doe', '555-1234', 1, 4.5, 'Available', 'Active'),
(112, 'Jane Smith', '555-5678', 2, 4.7, 'Available', 'Active'),
(113, 'Sam Wilson', '555-8765', 3, 4.2, 'Unavailable', 'Suspended'),
(114, 'Emily Clark', '555-2345', 4, 4.8, 'Available', 'Active'),
(115, 'Michael Brown', '555-3456', 5, 4.6, 'Unavailable', 'Banned'),
(116, 'Sarah Johnson', '555-6543', 6, 4.9, 'Available', 'Active'),
(117, 'David Lee', '555-7654', 7, 4.3, 'Available', 'Active'),
(118, 'Laura Davis', '555-4321', 8, 4.4, 'Unavailable', 'Suspended'),
(119, 'James Taylor', '555-8760', 9, 4.1, 'Available', 'Active'),
(120, 'Maria Gonzalez', '555-9876', 10, 4.7, 'Unavailable', 'Banned'),
(121, 'Robert Martinez', '555-5432', 11, 4.5, 'Available', 'Active'),
(122, 'Linda Wilson', '555-6542', 12, 3.6, 'Unavailable', 'Suspended'),
(123, 'Charles Moore', '555-7655', 13, 4.2, 'Available', 'Active'),
(124, 'Patricia Miller', '555-1235', 14, 4.8, 'Unavailable', 'Banned'),
(125, 'Joseph Jackson', '555-8761', 15, 4.9, 'Available', 'Active'),
(126, 'Thomas Harris', '555-4322', 16, 4.3, 'Unavailable', 'Suspended'),
(127, 'Nancy Clark', '555-9877', 17, 4.6, 'Available', 'Active'),
(128, 'Daniel Lewis', '555-5433', 18, 4.4, 'Unavailable', 'Banned'),
(129, 'Betty Walker', '555-6544', 19, 4.7, 'Available', 'Active'),
(130, 'Steven Allen', '555-7656', 20, 3.2, 'Unavailable', 'Suspended'),
(131, 'Ashley Young', '555-1236', 21, 4.8, 'Available', 'Active'),
(132, 'Jason King', '555-8762', 22, 4.3, 'Unavailable', 'Banned'),
(133, 'Dorothy Scott', '555-4323', 23, 4.5, 'Available', 'Active'),
(134, 'Kenneth Green', '555-9878', 24, 4.6, 'Unavailable', 'Suspended'),
(135, 'Jessica Adams', '555-5434', 25, 4.7, 'Available', 'Active'),
(136, 'Michael Nelson', '555-6545', 26, 2.2, 'Unavailable', 'Banned'),
(137, 'Frank Robinson', '555-7657', 27, 4.4, 'Available', 'Active'),
(138, 'Deborah Carter', '555-1237', 28, 3.8, 'Unavailable', 'Suspended'),
(139, 'Paul Perez', '555-8763', 29, 4.5, 'Available', 'Active'),
(140, 'Helen Mitchell', '555-4324', 30, 4.6, 'Unavailable', 'Banned');

"""
# Using try-except block to handle errors
try:
    # Connect to the database and execute the query
    with engine.connect() as conn:
        conn.execute(text(insert_query_driver))
        conn.commit()  # Commit the transaction
    print("Records inserted successfully into the Driver table.")
except Exception as e:
    # Capture any general exceptions
    print(f"An unexpected error occurred: {e}")

Records inserted successfully into the Driver table.


In [20]:
# Fetch all records from the Driver table
fetch_query = "SELECT * FROM Driver"
driver_data = fetch_data(fetch_query)

# Display the fetched records
if driver_data is not None:
    display(driver_data)
else:
    print("Failed to fetch data.")

,DriverID,Name,Contact,VehicleID,AvgDriverRating,AvailabilityStatus,DriverAccountStatus
0,111,John Doe,555-1234,1,4.5,Available,Active
1,112,Jane Smith,555-5678,2,4.7,Available,Active
2,113,Sam Wilson,555-8765,3,4.2,Unavailable,Suspended
3,114,Emily Clark,555-2345,4,4.8,Available,Active
4,115,Michael Brown,555-3456,5,4.6,Unavailable,Banned
5,116,Sarah Johnson,555-6543,6,4.9,Available,Active
6,117,David Lee,555-7654,7,4.3,Available,Active
7,118,Laura Davis,555-4321,8,4.4,Unavailable,Suspended
8,119,James Taylor,555-8760,9,4.1,Available,Active
9,120,Maria Gonzalez,555-9876,10,4.7,Unavailable,Banned


In [21]:
insert_query_trip_booking = """
INSERT INTO TripBooking (TripID, DriverID, PassengerID, VehicleID, PickupLocation, DropLocation, 
                         TripStatus, TripBookingTime, RideStartTime, RideEndTime, Fare, Tip)
VALUES 
(1001, 111, 11, 1, 'Washington Blvd', 'New York St', 'InProgress', '2021-02-11 19:12:22', '2021-02-11 19:22:22', '2021-02-11 19:52:22', 40.08, 10.08),
(1002, 112, 12, 2, 'Main St', 'Park Ave', 'Completed', '2021-02-11 20:12:22', '2021-02-11 20:22:22', '2021-02-11 20:52:22', 35.50, 8.00),
(1003, 113, 13, 3, 'Broadway', '5th Ave', 'Pending', '2021-02-11 21:12:22', '2021-02-11 21:22:22', '2021-02-11 21:52:22', 50.00, 12.00),
(1004, 114, 14, 4, 'Sunset Blvd', 'Ocean Drive', 'Completed', '2021-02-11 22:12:22', '2021-02-11 22:22:22', '2021-02-11 22:52:22', 55.50, 15.00),
(1005, 115, 15, 5, 'Hollywood Blvd', 'Sunset Blvd', 'InProgress', '2021-02-11 23:12:22', '2021-02-11 23:22:22', '2021-02-11 23:52:22', 45.20, 9.50),
(1006, 116, 16, 6, 'Pico Blvd', 'Santa Monica Blvd', 'Completed', '2021-02-12 00:12:22', '2021-02-12 00:22:22', '2021-02-12 00:52:22', 30.75, 5.00),
(1007, 117, 17, 7, 'Rose Ave', 'Cedar St', 'Pending', '2021-02-12 01:12:22', '2021-02-12 01:22:22', '2021-02-12 01:52:22', 40.00, 10.50),
(1008, 118, 18, 8, 'Lemon Dr', 'Elm St', 'InProgress', '2021-02-12 02:12:22', '2021-02-12 02:22:22', '2021-02-12 02:52:22', 32.20, 7.75),
(1009, 119, 19, 9, 'Cherry Ln', 'Birch Blvd', 'Completed', '2021-02-12 03:12:22', '2021-02-12 03:22:22', '2021-02-12 03:52:22', 38.45, 8.30),
(1010, 120, 20, 10, 'Oak St', 'Maple Ave', 'Pending', '2021-02-12 04:12:22', '2021-02-12 04:22:22', '2021-02-12 04:52:22', 55.30, 12.50),
(1011, 121, 21, 11, 'Palm Blvd', 'Pine Ave', 'Completed', '2021-02-12 05:12:22', '2021-02-12 05:22:22', '2021-02-12 05:52:22', 60.00, 16.00),
(1012, 122, 22, 12, 'King St', 'Queen St', 'InProgress', '2021-02-12 06:12:22', '2021-02-12 06:22:22', '2021-02-12 06:52:22', 48.90, 11.30),
(1013, 123, 23, 13, 'Broadway Ave', 'King St', 'Completed', '2021-02-12 07:12:22', '2021-02-12 07:22:22', '2021-02-12 07:52:22', 42.75, 9.00),
(1014, 124, 24, 14, 'Sunset Blvd', 'Vine St', 'Pending', '2021-02-12 08:12:22', '2021-02-12 08:22:22', '2021-02-12 08:52:22', 38.50, 7.20),
(1015, 125, 25, 15, 'Pacific Ave', 'Ocean Blvd', 'InProgress', '2021-02-12 09:12:22', '2021-02-12 09:22:22', '2021-02-12 09:52:22', 52.10, 13.60),
(1016, 126, 26, 16, 'First St', 'Second Ave', 'Completed', '2021-02-12 10:12:22', '2021-02-12 10:22:22', '2021-02-12 10:52:22', 29.30, 5.40),
(1017, 127, 27, 17, 'River Rd', 'Lake St', 'Pending', '2021-02-12 11:12:22', '2021-02-12 11:22:22', '2021-02-12 11:52:22', 63.40, 17.20),
(1018, 128, 28, 18, 'Elm Ave', 'Willow St', 'InProgress', '2021-02-12 12:12:22', '2021-02-12 12:22:22', '2021-02-12 12:52:22', 41.75, 9.80),
(1019, 129, 29, 19, 'Lincoln Blvd', 'Jefferson St', 'Completed', '2021-02-12 13:12:22', '2021-02-12 13:22:22', '2021-02-12 13:52:22', 50.10, 14.00),
(1020, 130, 30, 20, 'Maple Rd', 'Cedar Blvd', 'Pending', '2021-02-12 14:12:22', '2021-02-12 14:22:22', '2021-02-12 14:52:22', 44.20, 10.00),
(1021, 131, 31, 21, 'Park Ave', 'Rose Blvd', 'InProgress', '2021-02-12 15:12:22', '2021-02-12 15:22:22', '2021-02-12 15:52:22', 36.80, 8.50),
(1022, 132, 32, 22, 'Main Blvd', 'River St', 'Completed', '2021-02-12 16:12:22', '2021-02-12 16:22:22', '2021-02-12 16:52:22', 47.60, 11.40),
(1023, 133, 33, 23, 'Fifth Ave', 'Sixth St', 'Pending', '2021-02-12 17:12:22', '2021-02-12 17:22:22', '2021-02-12 17:52:22', 53.10, 12.90),
(1024, 134, 34, 24, 'Broadway Blvd', 'Sunset Rd', 'InProgress', '2021-02-12 18:12:22', '2021-02-12 18:22:22', '2021-02-12 18:52:22', 49.25, 10.60),
(1025, 135, 35, 25, 'Victory Blvd', 'Sunset Dr', 'Completed', '2021-02-12 19:12:22', '2021-02-12 19:22:22', '2021-02-12 19:52:22', 41.00, 9.20),
(1026, 136, 36, 26, 'Pacific Blvd', 'Ocean Ave', 'Pending', '2021-02-12 20:12:22', '2021-02-12 20:22:22', '2021-02-12 20:52:22', 57.90, 14.50),
(1027, 137, 37, 27, 'West St', 'East Blvd', 'InProgress', '2021-02-12 21:12:22', '2021-02-12 21:22:22', '2021-02-12 21:52:22', 33.80, 7.90),
(1028, 138, 38, 28, 'Hill Rd', 'Valley St', 'Completed', '2021-02-12 22:12:22', '2021-02-12 22:22:22', '2021-02-12 22:52:22', 39.60, 8.20),
(1029, 139, 39, 29, 'Harbor Blvd', 'Ocean Dr', 'Pending', '2021-02-12 23:12:22', '2021-02-12 23:22:22', '2021-02-12 23:52:22', 61.10, 16.30),
(1030, 140, 40, 30, 'Riverside Dr', 'Main St', 'InProgress', '2021-02-13 00:12:22', '2021-02-13 00:22:22', '2021-02-13 00:52:22', 45.50, 11.10);
"""

# Using try-except block to handle errors
try:
    # Connect to the database and execute the query
    with engine.connect() as conn:
        conn.execute(text(insert_query_trip_booking))
        conn.commit()  # Commit the transaction
    print("Records inserted successfully into the TripBooking table.")
except Exception as e:
    # Capture any general exceptions
    print(f"An unexpected error occurred: {e}")


Records inserted successfully into the TripBooking table.


In [22]:
# Fetch all records from the TripBooking table
fetch_query = "SELECT * FROM TripBooking"
tripBooking_data = fetch_data(fetch_query)

# Display the fetched records
if tripBooking_data is not None:
    display(tripBooking_data)
else:
    print("Failed to fetch data.")

,TripID,DriverID,PassengerID,VehicleID,PickupLocation,DropLocation,TripStatus,TripBookingTime,RideStartTime,RideEndTime,Fare,Tip
0,1001,111,11,1,Washington Blvd,New York St,InProgress,2021-02-11 19:12:22,2021-02-11 19:22:22,2021-02-11 19:52:22,40.08,10.08
1,1002,112,12,2,Main St,Park Ave,Completed,2021-02-11 20:12:22,2021-02-11 20:22:22,2021-02-11 20:52:22,35.50,8.00
2,1003,113,13,3,Broadway,5th Ave,Pending,2021-02-11 21:12:22,2021-02-11 21:22:22,2021-02-11 21:52:22,50.00,12.00
3,1004,114,14,4,Sunset Blvd,Ocean Drive,Completed,2021-02-11 22:12:22,2021-02-11 22:22:22,2021-02-11 22:52:22,55.50,15.00
4,1005,115,15,5,Hollywood Blvd,Sunset Blvd,InProgress,2021-02-11 23:12:22,2021-02-11 23:22:22,2021-02-11 23:52:22,45.20,9.50
5,1006,116,16,6,Pico Blvd,Santa Monica Blvd,Completed,2021-02-12 00:12:22,2021-02-12 00:22:22,2021-02-12 00:52:22,30.75,5.00
6,1007,117,17,7,Rose Ave,Cedar St,Pending,2021-02-12 01:12:22,2021-02-12 01:22:22,2021-02-12 01:52:22,40.00,10.50
7,1008,118,18,8,Lemon Dr,Elm St,InProgress,2021-02-12 02:12:22,2021-02-12 02:22:22,2021-02-12 02:52:22,32.20,7.75
8,1009,119,19,9,Cherry Ln,Birch Blvd,Completed,2021-02-12 03:12:22,2021-02-12 03:22:22,2021-02-12 03:52:22,38.45,8.30
9,1010,120,20,10,Oak St,Maple Ave,Pending,2021-02-12 04:12:22,2021-02-12 04:22:22,2021-02-12 04:52:22,55.30,12.50


In [23]:
insert_query_ride_history = """
INSERT INTO RideHistory (HistoryID, TripID, RideStatus)
VALUES
(11111, 1001, 'InProgress'),
(11112, 1002, 'Completed'),
(11113, 1003, 'Pending'),
(11114, 1004, 'Completed'),
(11115, 1005, 'InProgress'),
(11116, 1006, 'Completed'),
(11117, 1007, 'Pending'),
(11118, 1008, 'InProgress'),
(11119, 1009, 'Completed'),
(11120, 1010, 'InProgress'),
(11121, 1011, 'Completed'),
(11122, 1012, 'Pending'),
(11123, 1013, 'Completed'),
(11124, 1014, 'InProgress'),
(11125, 1015, 'Completed'),
(11126, 1016, 'InProgress'),
(11127, 1017, 'Completed'),
(11128, 1018, 'Pending'),
(11129, 1019, 'InProgress'),
(11130, 1020, 'Completed'),
(11131, 1021, 'Pending'),
(11132, 1022, 'Completed'),
(11133, 1023, 'InProgress'),
(11134, 1024, 'Completed'),
(11135, 1025, 'Pending'),
(11136, 1026, 'Completed'),
(11137, 1027, 'InProgress'),
(11138, 1028, 'Pending'),
(11139, 1029, 'Completed'),
(11140, 1030, 'InProgress');
"""

# Using try-except block to handle errors
try:
    # Connect to the database and execute the query
    with engine.connect() as conn:
        conn.execute(text(insert_query_ride_history))
        conn.commit()  # Commit the transaction
    print("Records inserted successfully into the RideHistory table.")
except Exception as e:
    # Capture any general exceptions
    print(f"An unexpected error occurred: {e}")


Records inserted successfully into the RideHistory table.


In [24]:
# Fetch all records from the RideHistory table
fetch_query = "SELECT * FROM RideHistory"
ridehistory_data = fetch_data(fetch_query)

# Display the fetched records
if ridehistory_data is not None:
    display(ridehistory_data)
else:
    print("Failed to fetch data.")

,HistoryID,TripID,RideStatus
0,11111,1001,InProgress
1,11112,1002,Completed
2,11113,1003,Pending
3,11114,1004,Completed
4,11115,1005,InProgress
5,11116,1006,Completed
6,11117,1007,Pending
7,11118,1008,InProgress
8,11119,1009,Completed
9,11120,1010,InProgress


In [25]:
 # SQL INSERT query
insert_query_rating = """
        INSERT INTO Rating (RatingID, DriverID, PassengerID, TripID, RatingScore, Feedback)
        VALUES
(101, 111, 11, 1001, 4.9, 'The driver is experienced and dropped me on time.'),
(102, 112, 12, 1002, 4.8, 'Great service, but the car could have been cleaner.'),
(103, 113, 13, 1003, 3.5, 'The ride was okay, but the driver took a wrong turn.'),
(104, 114, 14, 1004, 5.0, 'Excellent ride, very professional driver!'),
(105, 115, 15, 1005, 4.2, 'Good ride, but there was a delay due to traffic.'),
(106, 116, 16, 1006, 4.7, 'Very pleasant experience, would ride again!'),
(107, 117, 17, 1007, 4.6, 'Smooth ride, but the air conditioning was a bit too cold.'),
(108, 118, 18, 1008, 4.9, 'Driver was friendly and arrived ahead of time.'),
(109, 119, 19, 1009, 3.8, 'The ride was fine, but there were some route detours I was not expect.'),
(110, 120, 20, 1010, 5.0, 'Great ride, driver was professional and made great conversation.'),
(111, 121, 21, 1011, 4.4, 'Driver was good, but the car could have been more comfortable.'),
(112, 122, 22, 1012, 4.5, 'Nice drive, though the music was a bit too loud for my liking.'),
(113, 123, 23, 1013, 4.8, 'Excellent service and a clean car, very satisfied.'),
(114, 124, 24, 1014, 3.9, 'Good ride overall, but the driver was slightly late picking me up.'),
(115, 125, 25, 1015, 4.7, 'Comfortable ride, and the driver was polite and professional.'),
(116, 126, 26, 1016, 4.3, 'The ride was fine, though the driver could have taken a faster route.'),
(117, 127, 27, 1017, 5.0, 'Amazing service, the driver was prompt and friendly!'),
(118, 128, 28, 1018, 4.2, 'The ride was okay, but I wish the car was cleaner.'),
(119, 129, 29, 1019, 4.6, 'Good ride overall, the driver was very professional and courteous.'),
(120, 130, 30, 1020, 4.1, 'Driver was nice, but the ride was longer than expected due to traffic.'),
(121, 131, 31, 1021, 4.7, 'Comfortable ride, driver was very accommodating and helpful.'),
(122, 132, 32, 1022, 5.0, 'Best ride I had, the driver was exceptional'),
(123, 133, 33, 1023, 4.8, 'Fantastic ride, the driver knew all the best routes.'),
(124, 134, 34, 1024, 4.3, 'The ride was good, but the driver could have been more engaging.'),
(125, 135, 35, 1025, 4.9, 'I had a pleasant ride, and the driver was on time and polite.'),
(126, 136, 36, 1026, 4.0, 'Ride was fine, but the driver seemed distracted at times.'),
(127, 137, 37, 1027, 5.0, 'Excellent ride, very smooth and the driver was exceptional!'),
(128, 138, 38, 1028, 4.6, 'Comfortable and clean car, but I would have preferred a quieter ride.'),
(129, 139, 39, 1029, 4.2, 'The driver was good, but there was some confusion at the pickup point.'),
(130, 140, 40, 1030, 4.7, 'Great experience, the driver was professional and prompt.');
        """
        
# Using try-except block to handle errors
try:
    # Connect to the database and execute the query
    with engine.connect() as conn:
        conn.execute(text(insert_query_rating))
        conn.commit()  # Commit the transaction
    print("Records inserted successfully into the Rating table.")
except Exception as e:
    # Capture any general exceptions
    print(f"An unexpected error occurred: {e}")

Records inserted successfully into the Rating table.


In [26]:
# Fetch all records from the Rating table
fetch_query = "SELECT * FROM Rating"
rating_data = fetch_data(fetch_query)

# Display the fetched records
if rating_data is not None:
    display(rating_data)
else:
    print("Failed to fetch data.")

,RatingID,DriverID,PassengerID,TripID,RatingScore,Feedback
0,101,111,11,1001,4.9,The driver is experienced and dropped me on time.
1,102,112,12,1002,4.8,"Great service, but the car could have been cle..."
2,103,113,13,1003,3.5,"The ride was okay, but the driver took a wrong..."
3,104,114,14,1004,5.0,"Excellent ride, very professional driver!"
4,105,115,15,1005,4.2,"Good ride, but there was a delay due to traffic."
5,106,116,16,1006,4.7,"Very pleasant experience, would ride again!"
6,107,117,17,1007,4.6,"Smooth ride, but the air conditioning was a bi..."
7,108,118,18,1008,4.9,Driver was friendly and arrived ahead of time.
8,109,119,19,1009,3.8,"The ride was fine, but there were some route d..."
9,110,120,20,1010,5.0,"Great ride, driver was professional and made g..."


In [27]:
insert_query_payment = """
INSERT INTO Payment (PaymentID, TripID, PassengerID, PaymentMethod, PaymentStatus, Amount)
VALUES
(21, 1001, 11, 'Credit', 'Completed', 40.08),
(22, 1002, 12, 'Credit', 'Completed', 35.50),
(23, 1003, 13, 'Debit', 'Pending', 50.00),
(24, 1004, 14, 'Debit', 'Completed', 55.50),
(25, 1005, 15, 'Credit', 'Pending', 45.20),
(26, 1006, 16, 'Debit', 'Completed', 30.75),
(27, 1007, 17, 'Credit', 'Completed', 100.00),
(28, 1008, 18, 'Debit', 'Pending', 20.40),
(29, 1009, 19, 'Credit', 'Completed', 60.00),
(30, 1010, 20, 'Debit', 'Completed', 70.25),
(31, 1011, 21, 'Credit', 'Pending', 80.50),
(32, 1012, 22, 'Debit', 'Completed', 90.75),
(33, 1013, 23, 'Credit', 'Completed', 55.00),
(34, 1014, 24, 'Debit', 'Pending', 65.40),
(35, 1015, 25, 'Credit', 'Completed', 72.30),
(36, 1016, 26, 'Debit', 'Pending', 48.60),
(37, 1017, 27, 'Credit', 'Completed', 85.20),
(38, 1018, 28, 'Debit', 'Completed', 100.50),
(39, 1019, 29, 'Credit', 'Pending', 95.00),
(40, 1020, 30, 'Debit', 'Completed', 110.75),
(41, 1021, 31, 'Credit', 'Completed', 120.00),
(42, 1022, 32, 'Debit', 'Pending', 40.10),
(43, 1023, 33, 'Credit', 'Completed', 75.80),
(44, 1024, 34, 'Debit', 'Completed', 65.20),
(45, 1025, 35, 'Credit', 'Pending', 32.90),
(46, 1026, 36, 'Debit', 'Completed', 40.50),
(47, 1027, 37, 'Credit', 'Pending', 50.00),
(48, 1028, 38, 'Debit', 'Completed', 60.00),
(49, 1029, 39, 'Credit', 'Completed', 42.30),
(50, 1030, 40, 'Debit', 'Pending', 35.60);
"""

# Using try-except block to handle errors
try:
    # Connect to the database and execute the query
    with engine.connect() as conn:
        conn.execute(text(insert_query_payment))
        conn.commit()  # Commit the transaction
    print("Records inserted successfully into the Payment table.")
except Exception as e:
    # Capture any general exceptions
    print(f"An unexpected error occurred: {e}")


Records inserted successfully into the Payment table.


In [28]:
# Fetch all records from the Rating table
fetch_query = "SELECT * FROM Payment"
payment_data = fetch_data(fetch_query)

# Display the fetched records
if rating_data is not None:
    display(payment_data)
else:
    print("Failed to fetch data.")

,PaymentID,TripID,PassengerID,PaymentMethod,PaymentStatus,Amount
0,21,1001,11,Credit,Completed,40.08
1,22,1002,12,Credit,Completed,35.50
2,23,1003,13,Debit,Pending,50.00
3,24,1004,14,Debit,Completed,55.50
4,25,1005,15,Credit,Pending,45.20
5,26,1006,16,Debit,Completed,30.75
6,27,1007,17,Credit,Completed,100.00
7,28,1008,18,Debit,Pending,20.40
8,29,1009,19,Credit,Completed,60.00
9,30,1010,20,Debit,Completed,70.25


## One driver can accept no more than one trip at a time.

In [29]:
import pandas as pd

def check_driver_availability(driver_id, engine):
    # First, check the driver's account status
    check_driver_status_query = """
    SELECT DriverAccountStatus 
    FROM Driver 
    WHERE DriverID = %s;
    """
    
    # Execute the query to get the account status
    account_status_result = pd.read_sql(check_driver_status_query, engine, params=(driver_id,))
    
    # If no record is found for the given driver_id, return an error message
    if account_status_result.empty:
        return f"Driver {driver_id} not found."
    
    account_status = account_status_result.iloc[0, 0]
    
    # Check if the driver's account is active (modify 'Active' based on your account status values)
    if account_status != 'Active':
        return f"Driver {driver_id}'s account is not active. Cannot accept new bookings."
    
    # Now check if the driver already has an ongoing trip
    check_driver_availability_query = """
    SELECT COUNT(*) 
    FROM TripBooking 
    WHERE DriverID = %s
      AND TripStatus = 'InProgress';
    """
    
    # Execute the query to check ongoing trips
    result = pd.read_sql(check_driver_availability_query, engine, params=(driver_id,))
    
    # If result indicates an ongoing trip, return that the driver already has one
    if result.iloc[0, 0] > 0:
        return f"Driver {driver_id} is already on another trip."
    else:
        return f"Driver {driver_id} is available for booking."


In [30]:
driver_id = 115
result_message = check_driver_availability(driver_id, engine)
print(result_message)


Driver 115's account is not active. Cannot accept new bookings.


## A passenger can only book one trip at a time

In [31]:
import pandas as pd

def check_passenger_trip_by_account_status(passenger_id, engine):
    # First, check the passenger's account status
    check_account_status_query = """
    SELECT PassengerAccountStatus 
    FROM Passenger 
    WHERE PassengerID = %s;
    """
    
    # Execute the query to get the account status
    account_status_result = pd.read_sql(check_account_status_query, engine, params=(passenger_id,))
    
    # If no record found for the given passenger_id, return an error message
    if account_status_result.empty:
        return f"Passenger {passenger_id} not found."
    
    account_status = account_status_result.iloc[0, 0]
    
    # Check if the passenger account is active (modify 'Active' based on your account status values)
    if account_status != 'Active':
        return f"Passenger {passenger_id}'s account is not active. Cannot book a new trip."
    
    # Now check if the passenger already has an ongoing trip
    check_passenger_trip_query = """
    SELECT COUNT(*) 
    FROM TripBooking 
    WHERE PassengerID = %s 
      AND TripStatus = 'InProgress';
    """
    
    # Execute the query to check ongoing trips
    result = pd.read_sql(check_passenger_trip_query, engine, params=(passenger_id,))
    
    # If result indicates an ongoing trip, return that the passenger already has one
    if result.iloc[0, 0] > 0:
        return f"Passenger {passenger_id} already has an ongoing trip."
    else:
        return f"Passenger {passenger_id} can book a new trip."


passenger_id = 16
result_message = check_passenger_trip_by_account_status(passenger_id, engine)
print(result_message)


Passenger 16 can book a new trip.


## A ride will be only considered complete when the driver marks the ride as complete

In [32]:
from sqlalchemy import text
 
# Assuming engine is already defined as your database engine
update_query = """
UPDATE TripBooking
SET TripStatus = 'Completed', RideEndTime = CURRENT_TIMESTAMP
WHERE TripID = :trip_id AND DriverID = :driver_id AND TripStatus = 'InProgress';
"""
 
check_query = """
SELECT COUNT(*) FROM TripBooking
WHERE TripID = :trip_id AND DriverID = :driver_id;
"""
 
# Example values for trip_id and driver_id, you can get these dynamically from the user input
trip_id = 1001  # Example TripID
driver_id = 111  # Example DriverID
 
# Using try-except block to handle errors
try:
    with engine.connect() as conn:
        # First, check if the TripID and DriverID exist in the table
        result = conn.execute(text(check_query), {'trip_id': trip_id, 'driver_id': driver_id}).fetchone()
        if result[0] == 0:
            # If no matching rows found, print Invalid TripID
            print("Invalid TripID or DriverID.")
        else:
            # If a match is found, update the trip status
            conn.execute(text(update_query), {'trip_id': trip_id, 'driver_id': driver_id})
            conn.commit()  # Commit the transaction
            print(f"Trip {trip_id} marked as completed successfully.")
except Exception as e:
    # Capture any general exceptions
    print(f"An unexpected error occurred: {e}")

Trip 1001 marked as completed successfully.


## Payment is processed only after the ride has completed

In [33]:
import pandas as pd

def check_payment_for_completed_rides(engine):
    # Query to check for completed rides and processed payments
    check_payment_query = """
    SELECT p.PaymentID, p.TripID, p.PassengerID, p.PaymentMethod, p.PaymentStatus, p.Amount
    FROM Payment p
    JOIN TripBooking t ON p.TripID = t.TripID
    WHERE t.TripStatus = 'Completed'
      AND p.PaymentStatus = 'Completed';
    """
    
    # Execute the query to get the completed payment records
    result = pd.read_sql(check_payment_query, engine)
    
    # If no results are found, return an appropriate message
    if result.empty:
        return "No completed payments found for completed trips."
    
    # If results are found, return the details of completed payments
    return result

# Example usage
# Assuming 'engine' is your database connection object
completed_payments = check_payment_for_completed_rides(engine)
print(completed_payments)


   PaymentID  TripID  PassengerID PaymentMethod PaymentStatus  Amount
0         21    1001           11        Credit     Completed   40.08
1         22    1002           12        Credit     Completed   35.50
2         24    1004           14         Debit     Completed   55.50
3         26    1006           16         Debit     Completed   30.75
4         29    1009           19        Credit     Completed   60.00
5         33    1013           23        Credit     Completed   55.00
6         48    1028           38         Debit     Completed   60.00


In [34]:
import pandas as pd

def validate_payment_business_rule(engine):
    """
    Validate the business rule: Payment is processed only after the ride is completed.
    """
    # Query to check for any violations of the business rule
    check_payment_query = """
    SELECT 
        p.PaymentID, 
        p.TripID, 
        t.TripStatus, 
        p.PaymentStatus, 
        p.Amount
    FROM Payment p
    JOIN TripBooking t ON p.TripID = t.TripID
    WHERE p.PaymentStatus = 'Completed' AND t.TripStatus != 'Completed';
    """
    # Execute the query to find violations
    violations = pd.read_sql(check_payment_query, engine)
    
    # If violations exist, return them
    if not violations.empty:
        return f"Payments processed for incomplete trips:\n{violations}"
    
    # If no violations exist, return a success message
    return "All completed payments correspond to completed trips only."

# Example usage
# Assuming 'engine' is your database connection object
validation_result = validate_payment_business_rule(engine)
print(validation_result)


Payments processed for incomplete trips:
    PaymentID  TripID  TripStatus PaymentStatus  Amount
0          27    1007     Pending     Completed  100.00
1          30    1010     Pending     Completed   70.25
2          32    1012  InProgress     Completed   90.75
3          35    1015  InProgress     Completed   72.30
4          37    1017     Pending     Completed   85.20
5          38    1018  InProgress     Completed  100.50
6          40    1020     Pending     Completed  110.75
7          41    1021  InProgress     Completed  120.00
8          43    1023     Pending     Completed   75.80
9          44    1024  InProgress     Completed   65.20
10         46    1026     Pending     Completed   40.50
11         49    1029     Pending     Completed   42.30


In [35]:

# SQL for creating the trigger
trigger_code = """
CREATE TRIGGER enforce_trip_completion_before_payment
BEFORE UPDATE ON Payment
FOR EACH ROW
BEGIN
    DECLARE trip_status VARCHAR(50);

    -- Retrieve the associated trip status
    SELECT TripStatus INTO trip_status
    FROM TripBooking
    WHERE TripID = NEW.TripID;

    -- Raise an error if the trip is not completed
    IF NEW.PaymentStatus = 'Completed' AND trip_status != 'Completed' THEN
        SIGNAL SQLSTATE '45000'
        SET MESSAGE_TEXT = 'Payment cannot be marked as completed for incomplete trips.';
    END IF;
END;
"""

# Execute the SQL to create the trigger
with engine.connect() as conn:
    conn.execute(text(trigger_code))

print("Trigger created successfully.")

Trigger created successfully.


In [36]:
update_query = text("""
UPDATE Payment
SET PaymentStatus = 'Completed'
WHERE PaymentID = 21;
""")

try:
    with engine.connect() as conn:
        conn.execute(update_query)
except Exception as e:
    print("Trigger enforced:", e)


In [37]:
update_query = text("UPDATE Payment SET PaymentStatus = :status WHERE PaymentID = :id")
with engine.connect() as conn:
    conn.execute(update_query, {"status": "Completed", "id": 24})
    print("Payment status updated successfully.")


Payment status updated successfully.


## Drivers must maintain an average rating of 4.0 or more to continue being active on the platform.

In [38]:
from sqlalchemy import text
 
# Query to select drivers with an average rating below 5.0
select_low_rated_drivers_query = """
SELECT DriverID, Name, AvgDriverRating, DriverAccountStatus
FROM Driver
WHERE AvgDriverRating < 4.0;
"""
 
# Query to update the account status of drivers with an average rating below 5.0
update_driver_status_query = """
UPDATE Driver
SET DriverAccountStatus = 'Suspended'
WHERE AvgDriverRating < 4.0;
"""
 
# Using try-except block to handle errors
try:
    with engine.connect() as conn:
        # First, check the drivers who have a low rating
        result = conn.execute(text(select_low_rated_drivers_query)).fetchall()
        # If any drivers have a low rating
        if result:
            print("Drivers with a rating below 4.0:")
            # Loop over the result and access the columns using indexing
            for driver in result:
                # Access the values by index in the tuple (since SQLAlchemy returns results in tuples)
                driver_id = driver[0]  # DriverID
                name = driver[1]  # Name
                avg_rating = driver[2]  # AvgDriverRating
                account_status = driver[3]  # DriverAccountStatus
 
                # Print the driver information
                print(f"DriverID: {driver_id}, Name: {name}, Rating: {avg_rating}, Status: {account_status}")
            # Now, update the status for these drivers
            conn.execute(text(update_driver_status_query))
            conn.commit()  # Commit the transaction
            print("Account status for low-rated drivers updated to 'Suspended'.")
        else:
            print("No drivers found with a rating below 4.0.")
except Exception as e:
    # Capture any general exceptions
    print(f"An unexpected error occurred: {e}")

Drivers with a rating below 4.0:
DriverID: 122, Name: Linda Wilson, Rating: 3.6, Status: Suspended
DriverID: 130, Name: Steven Allen, Rating: 3.2, Status: Suspended
DriverID: 136, Name: Michael Nelson, Rating: 2.2, Status: Banned
DriverID: 138, Name: Deborah Carter, Rating: 3.8, Status: Suspended
Account status for low-rated drivers updated to 'Suspended'.


# After a trip, passengers rate drivers but have no access to a driver's information beyond what is needed to take the trip.

In [39]:
from sqlalchemy import text

# Query to select completed trips where passengers can rate the driver
# and also include the driver's ride rating from the Rating table
select_completed_trips_for_rating_query = """
SELECT t.TripID, t.DriverID, t.PassengerID, t.RideEndTime, r.RatingScore
FROM TripBooking t
JOIN Rating r ON t.TripID = r.TripID
WHERE t.TripStatus = 'Completed';
"""

# Using try-except block to handle errors
try:
    with engine.connect() as conn:
        # Step 1: Fetch completed trips with ratings
        result = conn.execute(text(select_completed_trips_for_rating_query)).fetchall()
        
        if result:
            # Step 2: Print the completed trips with driver ratings
            print("Completed Trips Eligible for Rating:")
            for trip in result:
                trip_id = trip[0]
                driver_id = trip[1]
                passenger_id = trip[2]
                ride_end_time = trip[3]
                rating_score = trip[4]

                # Printing the details of the completed trip along with the rating score
                print(f"TripID: {trip_id}, DriverID: {driver_id}, PassengerID: {passenger_id}, RideEndTime: {ride_end_time}, Driver Rating: {rating_score}")
        else:
            print("No completed trips found for ratings.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Completed Trips Eligible for Rating:
TripID: 1001, DriverID: 111, PassengerID: 11, RideEndTime: 2024-11-27 12:43:51, Driver Rating: 4.9
TripID: 1002, DriverID: 112, PassengerID: 12, RideEndTime: 2021-02-11 20:52:22, Driver Rating: 4.8
TripID: 1004, DriverID: 114, PassengerID: 14, RideEndTime: 2021-02-11 22:52:22, Driver Rating: 5.0
TripID: 1006, DriverID: 116, PassengerID: 16, RideEndTime: 2021-02-12 00:52:22, Driver Rating: 4.7
TripID: 1009, DriverID: 119, PassengerID: 19, RideEndTime: 2021-02-12 03:52:22, Driver Rating: 3.8
TripID: 1011, DriverID: 121, PassengerID: 21, RideEndTime: 2021-02-12 05:52:22, Driver Rating: 4.4
TripID: 1013, DriverID: 123, PassengerID: 23, RideEndTime: 2021-02-12 07:52:22, Driver Rating: 4.8
TripID: 1016, DriverID: 126, PassengerID: 26, RideEndTime: 2021-02-12 10:52:22, Driver Rating: 4.3
TripID: 1019, DriverID: 129, PassengerID: 29, RideEndTime: 2021-02-12 13:52:22, Driver Rating: 4.6
TripID: 1022, DriverID: 132, PassengerID: 32, RideEndTime: 2021-02-12 16

# Drivers do not have access to the passenger's details, except their name and the locations of pick-up and drop.

In [40]:
from sqlalchemy import text

# Function to get details of trips for a specific driver
def get_driver_trip_details(engine, driver_id):
    query = """
    SELECT 
        t.TripID, 
        t.DriverID, 
        p.Name AS PassengerName, 
        t.PickupLocation, 
        t.DropLocation
    FROM 
        TripBooking t
    JOIN 
        Passenger p ON t.PassengerID = p.PassengerID
    WHERE 
        t.DriverID = :driver_id;
    """
    
    try:
        # Execute the query and fetch the result
        with engine.connect() as conn:
            result = conn.execute(text(query), {'driver_id': driver_id}).fetchall()
        
        if result:
            # Print the trip details for the driver
            print(f"Trip details for Driver ID {driver_id}:")
            for row in result:
                trip_id, driver_id, passenger_name, pickup_location, drop_location = row
                print(f"TripID: {trip_id}, Passenger: {passenger_name}, Pickup: {pickup_location}, Drop: {drop_location}")
        else:
            print(f"No trips found for Driver ID {driver_id}.")
    
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


In [41]:
# Example usage with the engine and a driver ID
driver_id = 112  # Example Driver ID
get_driver_trip_details(engine, driver_id)

Trip details for Driver ID 112:
TripID: 1002, Passenger: Rachel Adams, Pickup: Main St, Drop: Park Ave


# Drivers may set their status regarding availability but must complete accepted trips before updating the status.

In [42]:
from sqlalchemy import text

def update_driver_availability(engine, driver_id, new_status):
    # SQL query to check if the driver has any "InProgress" or "Pending" trips
    check_trip_query = """
    SELECT COUNT(*)
    FROM TripBooking 
    WHERE DriverID = :driver_id
      AND TripStatus IN ('InProgress', 'Pending');
    """
    
    try:
        with engine.connect() as conn:
            # First, check if the driver has any ongoing trips
            result = conn.execute(text(check_trip_query), {'driver_id': driver_id}).fetchone()
            ongoing_trips = result[0]  # Get the count of ongoing trips
            
            if ongoing_trips == 0:
                # If no ongoing trips, update the driver's availability status
                update_query = """
                UPDATE Driver
                SET AvailabilityStatus = :new_status
                WHERE DriverID = :driver_id
                  AND NOT EXISTS (
                      SELECT 1
                      FROM TripBooking 
                      WHERE DriverID = :driver_id
                      AND TripStatus IN ('InProgress', 'Pending')
                  );
                """
                
                conn.execute(text(update_query), {'driver_id': driver_id, 'new_status': new_status})
                conn.commit()  # Commit the transaction
                
                print(f"Driver ID {driver_id} availability status updated to {new_status}.")
            else:
                print(f"Driver ID {driver_id} cannot update availability status because they have ongoing trips.")
    
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


In [43]:
# Example usage with the engine and a driver ID
driver_id = 113  # Example Driver ID
new_status = 'Unavailable'  # Example new availability status
update_driver_availability(engine, driver_id, new_status)

Driver ID 113 cannot update availability status because they have ongoing trips.


# If there are passenger cancellations, there is a fee incurred when cancelled in less than 5 minutes of the allotted time.

In [44]:
from sqlalchemy import text
import pandas as pd

def check_passenger_cancellations(engine):
    # SQL query to check for cancellations within 5 minutes of the trip booking time
    cancellation_query = """
    SELECT TripID, PassengerID, TripBookingTime, 
           CASE 
               WHEN TIMESTAMPDIFF(MINUTE, TripBookingTime, NOW()) <= 5 THEN 'Fee Applied'
               ELSE 'No Fee'
           END AS CancellationFeeStatus
    FROM TripBooking
    WHERE TripStatus = 'Cancelled'
      AND TIMESTAMPDIFF(MINUTE, TripBookingTime, NOW()) <= 5;
    """
    
    try:
        with engine.connect() as conn:
            # Execute the query
            result = pd.read_sql(cancellation_query, conn)
            
            # If there are cancellations, display them
            if not result.empty:
                print(result)
            else:
                print("No cancellations within 5 minutes found.")
    
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# Example usage with the engine
check_passenger_cancellations(engine)

No cancellations within 5 minutes found.


# Administrators can suspend or ban users from violating platform policies.

In [45]:
from sqlalchemy import text

def suspend_or_ban_user(engine, user_type, user_id, status):
    """
    Function to suspend or ban a user (Driver or Passenger) based on their ID.
    Parameters:
    - engine: The database engine (SQLAlchemy)
    - user_type: 'driver' or 'passenger' indicating the type of user
    - user_id: The ID of the user to suspend/ban
    - status: The status to set ('Suspended' or 'Banned')
    """
    
    if user_type == 'driver':
        update_query = """
        UPDATE Driver
        SET DriverAccountStatus = :status
        WHERE DriverID = :user_id;
        """
    elif user_type == 'passenger':
        update_query = """
        UPDATE Passenger
        SET PassengerAccountStatus = :status
        WHERE PassengerID = :user_id;
        """
    else:
        raise ValueError("Invalid user type. Use 'driver' or 'passenger'.")
    
    try:
        # Connect to the database and execute the query
        with engine.connect() as conn:
            conn.execute(text(update_query), {'status': status, 'user_id': user_id})
            conn.commit()  # Commit the transaction
        print(f"User {user_id} has been {status}.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

In [46]:
# Example usage:
# Suspend a driver with DriverID = 113
suspend_or_ban_user(engine, user_type='driver', user_id=113, status='Suspended')

# Ban a passenger with PassengerID = 12
suspend_or_ban_user(engine, user_type='passenger', user_id=12, status='Banned')

User 113 has been Suspended.
User 12 has been Banned.


In [47]:
from sqlalchemy import text
 
def suspend_or_ban_user(engine, user_type, user_id):
    """
    Function to automatically suspend or ban a user (Driver or Passenger) based on their current status.
    Parameters:
    - engine: The database engine (SQLAlchemy)
    - user_type: 'driver' or 'passenger' indicating the type of user
    - user_id: The ID of the user to suspend/ban
    """
    # Determine the query to check the current status of the user
    if user_type == 'driver':
        # Check the current status of the driver
        check_query = """
        SELECT DriverAccountStatus, AvailabilityStatus 
        FROM Driver
        WHERE DriverID = :user_id;
        """
    elif user_type == 'passenger':
        # Check the current status of the passenger
        check_query = """
        SELECT PassengerAccountStatus 
        FROM Passenger
        WHERE PassengerID = :user_id;
        """
    else:
        raise ValueError("Invalid user type. Use 'driver' or 'passenger'.")
    try:
        with engine.connect() as conn:
            # Fetch current status of the user
            result = conn.execute(text(check_query), {'user_id': user_id}).fetchone()
            if result:
                if user_type == 'driver':
                    driver_account_status, availability_status = result
                    # Apply logic for updating driver status
                    if driver_account_status == 'Active' and availability_status == 'Unavailable':
                        new_status = 'Suspended'
                    elif driver_account_status == 'Suspended' and availability_status == 'Unavailable':
                        new_status = 'Banned'
                    elif driver_account_status == 'Active':
                        new_status = 'Active'
                    else:
                        new_status = driver_account_status
 
                elif user_type == 'passenger':
                    passenger_account_status = result[0]
                    # Apply logic for updating passenger status
                    if passenger_account_status == 'Active':
                        # Assuming some condition where a passenger can be suspended or banned
                        # For example, a passenger could be banned for violating the platform's policy
                        new_status = 'Suspended'  # You can replace with actual violation logic if needed
                    else:
                        new_status = passenger_account_status
 
                # Now update the status based on the new status logic
                if user_type == 'driver':
                    update_query = """
                    UPDATE Driver
                    SET DriverAccountStatus = :status
                    WHERE DriverID = :user_id;
                    """
                elif user_type == 'passenger':
                    update_query = """
                    UPDATE Passenger
                    SET PassengerAccountStatus = :status
                    WHERE PassengerID = :user_id;
                    """
                # Update the user's account status
                conn.execute(text(update_query), {'status': new_status, 'user_id': user_id})
                conn.commit()  # Commit the transaction
 
                print(f"User {user_id} has been updated to {new_status}.")
            else:
                print(f"User with ID {user_id} does not exist.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
 
 
# Example usage:
# Suspend a driver with DriverID = 113 based on their current status
suspend_or_ban_user(engine, user_type='driver', user_id=113)
 
# Update the status of a passenger with PassengerID = 12 based on their current status
suspend_or_ban_user(engine, user_type='passenger', user_id=12)

User 113 has been updated to Banned.
User 12 has been updated to Banned.


In [48]:
from sqlalchemy import text
 
def print_driver_avg_rating(engine, driver_id):

    """

    Function to calculate and print the average rating of a specific driver

    based on ratings provided by passengers in the Rating table.

    Parameters:

    - engine: The SQLAlchemy engine connected to the database.

    - driver_id: The ID of the driver for whom the average rating is calculated.

    """

    try:

        # Query to calculate the average rating for a particular driver

        avg_rating_query = """

        SELECT AVG(RatingScore) AS AvgRating

        FROM Rating

        WHERE DriverID = :driver_id;

        """

        # Connect to the database and execute the query to get the average rating

        with engine.connect() as conn:

            result = conn.execute(text(avg_rating_query), {'driver_id': driver_id}).fetchone()

            if result and result[0] is not None:

                avg_rating = result[0]

                print(f"The average rating for driver {driver_id} is {avg_rating:.2f}.")

            else:

                print(f"No ratings found for driver {driver_id}.")

    except Exception as e:

        print(f"An unexpected error occurred: {e}")
 
# usage: Print the average rating for a driver with DriverID = 111

print_driver_avg_rating(engine, driver_id=111)
 
# Example usage: Print the average rating for a driver with DriverID = 112

print_driver_avg_rating(engine, driver_id=112)

 
 

The average rating for driver 111 is 4.90.
The average rating for driver 112 is 4.80.


# SQL Queries Using All Tables

## 1. Retrieve Trip Details with Associated Driver, Passenger, Vehicle, and Payment Information

In [49]:
import pandas as pd
from sqlalchemy import create_engine, text
# Assuming `engine` is already defined and connected to your MySQL database
# Define the query
query = """
SELECT 
    TB.TripID, 
    TB.PickupLocation, 
    TB.DropLocation, 
    D.Name AS DriverName, 
    P.Name AS PassengerName, 
    VD.Model AS VehicleModel, 
    VD.VehicleColor, 
    PM.Amount AS PaymentAmount
FROM 
    TripBooking TB
JOIN Driver D ON TB.DriverID = D.DriverID
JOIN Passenger P ON TB.PassengerID = P.PassengerID
JOIN VehicleDetails VD ON TB.VehicleID = VD.VehicleID
JOIN Payment PM ON TB.TripID = PM.TripID;
"""
# Execute the query
with engine.connect() as conn:
    result = conn.execute(text(query))
    # Convert the result to a Pandas DataFrame
    df = pd.DataFrame(result.fetchall(), columns=result.keys())
# Display the DataFrame
df


,TripID,PickupLocation,DropLocation,DriverName,PassengerName,VehicleModel,VehicleColor,PaymentAmount
0,1001,Washington Blvd,New York St,John Doe,David Lee,Corolla,Red,40.08
1,1002,Main St,Park Ave,Jane Smith,Rachel Adams,Honda Civic,Blue,35.50
2,1003,Broadway,5th Ave,Sam Wilson,Tom Harris,Ford Mustang,Black,50.00
3,1004,Sunset Blvd,Ocean Drive,Emily Clark,Sophia Green,Chevrolet Malibu,White,55.50
4,1005,Hollywood Blvd,Sunset Blvd,Michael Brown,James White,BMW 3 Series,Silver,45.20
5,1006,Pico Blvd,Santa Monica Blvd,Sarah Johnson,Olivia Black,Tesla Model 3,Red,30.75
6,1007,Rose Ave,Cedar St,David Lee,Emma Brown,Toyota Camry,Green,100.00
7,1008,Lemon Dr,Elm St,Laura Davis,Liam Smith,Honda Accord,Black,20.40
8,1009,Cherry Ln,Birch Blvd,James Taylor,Noah Johnson,Nissan Altima,Blue,60.00
9,1010,Oak St,Maple Ave,Maria Gonzalez,Ava Jones,Ford Focus,Red,70.25


## 2. List Completed Trips with Feedback and Ratings

In [50]:

# Define the query
query = """
SELECT 
    TB.TripID, 
    D.Name AS DriverName, 
    P.Name AS PassengerName, 
    RT.RatingScore as DriverRatingScore, 
    RT.Feedback
FROM 
    TripBooking TB
JOIN Driver D ON TB.DriverID = D.DriverID
JOIN Passenger P ON TB.PassengerID = P.PassengerID
JOIN Rating RT ON TB.TripID = RT.TripID
WHERE 
    TB.TripStatus = 'Completed';
"""

# Execute the query
with engine.connect() as conn:
    result = conn.execute(text(query))
    # Convert the result to a Pandas DataFrame
    df = pd.DataFrame(result.fetchall(), columns=result.keys())

# Display the DataFrame
df

,TripID,DriverName,PassengerName,DriverRatingScore,Feedback
0,1001,John Doe,David Lee,4.9,The driver is experienced and dropped me on time.
1,1002,Jane Smith,Rachel Adams,4.8,"Great service, but the car could have been cle..."
2,1004,Emily Clark,Sophia Green,5.0,"Excellent ride, very professional driver!"
3,1006,Sarah Johnson,Olivia Black,4.7,"Very pleasant experience, would ride again!"
4,1009,James Taylor,Noah Johnson,3.8,"The ride was fine, but there were some route d..."
5,1011,Robert Martinez,William Garcia,4.4,"Driver was good, but the car could have been m..."
6,1013,Charles Moore,Lucas Rodriguez,4.8,"Excellent service and a clean car, very satisf..."
7,1016,Thomas Harris,Charlotte Moore,4.3,"The ride was fine, though the driver could hav..."
8,1019,Betty Walker,Elijah Thomas,4.6,"Good ride overall, the driver was very profess..."
9,1022,Jason King,Sofia Lee,5.0,"Best ride I had, the driver was exceptional"


## 3. Find Total Earnings for Drivers, Grouped by Driver Name

In [51]:

# Define the query
query = """
SELECT 
    D.Name AS DriverName, 
    SUM(PM.Amount) AS TotalEarnings
FROM 
    Driver D
JOIN TripBooking TB ON D.DriverID = TB.DriverID
JOIN Payment PM ON TB.TripID = PM.TripID
GROUP BY 
    D.Name;
"""

# Execute the query
with engine.connect() as conn:
    result = conn.execute(text(query))
    # Convert the result to a Pandas DataFrame
    df = pd.DataFrame(result.fetchall(), columns=result.keys())

# Display the DataFrame
df

,DriverName,TotalEarnings
0,John Doe,40.08
1,Jane Smith,35.50
2,Sam Wilson,50.00
3,Emily Clark,55.50
4,Michael Brown,45.20
5,Sarah Johnson,30.75
6,David Lee,100.00
7,Laura Davis,20.40
8,James Taylor,60.00
9,Maria Gonzalez,70.25


## 4. Count Completed Trips for Each Vehicle

In [52]:

# Define the query
query = """
SELECT 
    VD.Model AS VehicleModel, 
    VD.LicensePlateNum, 
    COUNT(TB.TripID) AS CompletedTrips
FROM 
    VehicleDetails VD
JOIN TripBooking TB ON VD.VehicleID = TB.VehicleID
WHERE 
    TB.TripStatus = 'Completed'
GROUP BY 
    VD.Model, VD.LicensePlateNum;
"""

# Execute the query
with engine.connect() as conn:
    result = conn.execute(text(query))
    # Convert the result to a Pandas DataFrame
    df = pd.DataFrame(result.fetchall(), columns=result.keys())

# Display the DataFrame
df

,VehicleModel,LicensePlateNum,CompletedTrips
0,Corolla,TX53JJ3667,1
1,Honda Civic,TX45ZZ1234,1
2,Chevrolet Malibu,XY98KL2345,1
3,Tesla Model 3,GG32FF7890,1
4,Nissan Altima,MM33VV8765,1
5,Chevrolet Impala,QQ55XX6543,1
6,Mazda 3,SS77ZZ4321,1
7,Volkswagen Jetta,VV00CC1098,1
8,Mercedes-Benz,YY33FF8765,1
9,Porsche 911,BB66II5432,1


## 5. List Active Drivers with Their Average Ratings and Ride History Count

In [53]:

# Define the query
query = """
SELECT 
    D.Name AS DriverName, 
    D.AvgDriverRating, 
    COUNT(RH.HistoryID) AS RideCount
FROM 
    Driver D
JOIN TripBooking TB ON D.DriverID = TB.DriverID
JOIN RideHistory RH ON TB.TripID = RH.TripID
WHERE 
    D.DriverAccountStatus = 'Active'
GROUP BY 
    D.DriverID, D.Name, D.AvgDriverRating;
"""

# Execute the query
with engine.connect() as conn:
    result = conn.execute(text(query))
    # Convert the result to a Pandas DataFrame
    df = pd.DataFrame(result.fetchall(), columns=result.keys())

# Display the DataFrame
df

,DriverName,AvgDriverRating,RideCount
0,John Doe,4.5,1
1,Jane Smith,4.7,1
2,Emily Clark,4.8,1
3,Sarah Johnson,4.9,1
4,David Lee,4.3,1
5,James Taylor,4.1,1
6,Robert Martinez,4.5,1
7,Charles Moore,4.2,1
8,Joseph Jackson,4.9,1
9,Nancy Clark,4.6,1


## 6. Retrieve Trip Information with Payment and Rating Details

In [54]:

# Define the query
query = """
SELECT 
    TB.TripID, 
    TB.PickupLocation, 
    TB.DropLocation, 
    PM.Amount AS PaymentAmount, 
    RT.RatingScore, 
    RT.Feedback
FROM 
    TripBooking TB
JOIN Payment PM ON TB.TripID = PM.TripID
JOIN Rating RT ON TB.TripID = RT.TripID;
"""

# Execute the query
with engine.connect() as conn:
    result = conn.execute(text(query))
    # Convert the result to a Pandas DataFrame
    df = pd.DataFrame(result.fetchall(), columns=result.keys())

# Display the DataFrame
df

,TripID,PickupLocation,DropLocation,PaymentAmount,RatingScore,Feedback
0,1001,Washington Blvd,New York St,40.08,4.9,The driver is experienced and dropped me on time.
1,1002,Main St,Park Ave,35.50,4.8,"Great service, but the car could have been cle..."
2,1003,Broadway,5th Ave,50.00,3.5,"The ride was okay, but the driver took a wrong..."
3,1004,Sunset Blvd,Ocean Drive,55.50,5.0,"Excellent ride, very professional driver!"
4,1005,Hollywood Blvd,Sunset Blvd,45.20,4.2,"Good ride, but there was a delay due to traffic."
5,1006,Pico Blvd,Santa Monica Blvd,30.75,4.7,"Very pleasant experience, would ride again!"
6,1007,Rose Ave,Cedar St,100.00,4.6,"Smooth ride, but the air conditioning was a bi..."
7,1008,Lemon Dr,Elm St,20.40,4.9,Driver was friendly and arrived ahead of time.
8,1009,Cherry Ln,Birch Blvd,60.00,3.8,"The ride was fine, but there were some route d..."
9,1010,Oak St,Maple Ave,70.25,5.0,"Great ride, driver was professional and made g..."


## 7. Identify Drivers With Low Ratings and Their Completed Trips

In [55]:
# Define the query
query = """
SELECT 
    D.Name AS DriverName, 
    AVG(RT.RatingScore) AS AvgRating, 
    COUNT(TB.TripID) AS CompletedTrips
FROM 
    Driver D
JOIN TripBooking TB ON D.DriverID = TB.DriverID
JOIN Rating RT ON TB.TripID = RT.TripID
WHERE 
    RT.RatingScore < 4.0
GROUP BY 
    D.Name
HAVING 
    AVG(RT.RatingScore) < 4.0;
"""

# Execute the query
with engine.connect() as conn:
    result = conn.execute(text(query))
    # Convert the result to a Pandas DataFrame
    df = pd.DataFrame(result.fetchall(), columns=result.keys())

# Display the DataFrame
df

,DriverName,AvgRating,CompletedTrips
0,Sam Wilson,3.50000,1
1,James Taylor,3.80000,1
2,Patricia Miller,3.90000,1


## 8. Retrieve a Summary of Trips Grouped by Drivers and Payment Status



In [56]:
# Define the query
query = """
SELECT 
    D.Name AS DriverName, 
    PM.PaymentStatus, 
    COUNT(TB.TripID) AS TripCount, 
    SUM(PM.Amount) AS TotalEarnings
FROM 
    Driver D
JOIN TripBooking TB ON D.DriverID = TB.DriverID
JOIN Payment PM ON TB.TripID = PM.TripID
GROUP BY 
    D.Name, PM.PaymentStatus;
"""

# Execute the query
with engine.connect() as conn:
    result = conn.execute(text(query))
    # Convert the result to a Pandas DataFrame
    df = pd.DataFrame(result.fetchall(), columns=result.keys())

# Display the DataFrame
df

,DriverName,PaymentStatus,TripCount,TotalEarnings
0,John Doe,Completed,1,40.08
1,Jane Smith,Completed,1,35.50
2,Sam Wilson,Pending,1,50.00
3,Emily Clark,Completed,1,55.50
4,Michael Brown,Pending,1,45.20
5,Sarah Johnson,Completed,1,30.75
6,David Lee,Completed,1,100.00
7,Laura Davis,Pending,1,20.40
8,James Taylor,Completed,1,60.00
9,Maria Gonzalez,Completed,1,70.25
